# 🗂️ Module 07 — Tuples, Sets & Dictionaries

## 🎯 What you will learn
1. **Tuples** – immutable sequences, packing/unpacking, named tuples
2. **Sets** – uniqueness, set algebra, frozenset
3. **Dictionaries** – key/value mapping, every method, nested dicts, counting & grouping
4. Hashing: why keys must be hashable
5. Choosing the right collection (with big-O)
6. `collections` module: `Counter`, `defaultdict`, `OrderedDict`, `ChainMap`, `namedtuple`
7. Hidden facts & tricks

**Exam relevance:** ⭐⭐⭐⭐⭐ (comparison tables are asked every year)

---
# PART A — TUPLES

A tuple is an **ordered, immutable** sequence written with parentheses (the commas actually make the tuple).

In [1]:
t = (10, 20, 30)
single = (5,)          # comma required for a 1-item tuple!
not_a_tuple = (5)
no_parens = 1, 2, 3    # parentheses optional
empty = ()
print(t, type(single), type(not_a_tuple), no_parens, empty)

print(t[0], t[-1], t[1:], len(t), 20 in t)
print(t.count(10), t.index(30))
print(t + (40,), t * 2)

try:
    t[0] = 99
except TypeError as e:
    print("TypeError:", e)

(10, 20, 30) <class 'tuple'> <class 'int'> (1, 2, 3) ()
10 30 (20, 30) 3 True
1 2
(10, 20, 30, 40) (10, 20, 30, 10, 20, 30)
TypeError: 'tuple' object does not support item assignment


### Why use tuples if lists exist?
1. **Safety** – data that shouldn't change (coordinates, RGB, DB rows)
2. **Hashable** – can be dict keys / set members (if their contents are hashable)
3. **Faster & smaller** than lists
4. **Semantic** – tuple = a record of different things; list = a collection of similar things
5. Functions can return multiple values as a tuple

In [2]:
import sys
print(sys.getsizeof((1, 2, 3)), sys.getsizeof([1, 2, 3]))

def min_max(seq):
    return min(seq), max(seq)         # returns a tuple
lo, hi = min_max([4, 1, 9])
print(lo, hi)

# Packing / unpacking
point = (3, 4)
x, y = point
a, *middle, z = (1, 2, 3, 4, 5)
print(x, y, a, middle, z)
(p, q), r = (1, 2), 3                 # nested unpacking
print(p, q, r)

# Tuples as dict keys
grid = {(0, 0): "start", (2, 3): "goal"}
print(grid[(2, 3)])

# "Immutable" but can hold mutable items
weird = ([1, 2], 3)
weird[0].append(99)
print(weird)

64 88
1 9
3 4 1 [2, 3, 4] 5
1 2 3
goal
([1, 2, 99], 3)


### Named tuples — readable records

In [3]:
from collections import namedtuple
Point = namedtuple("Point", ["x", "y"])
p = Point(3, 4)
print(p, p.x, p[1], p._asdict())
p2 = p._replace(x=10)
print(p2)

Point(x=3, y=4) 3 4 {'x': 3, 'y': 4}
Point(x=10, y=4)


---
# PART B — SETS

A set is an **unordered collection of unique, hashable items**, written `{1, 2, 3}`. Empty set is `set()` – `{}` is an empty **dict**!

In [4]:
s = {3, 1, 2, 3, 3}
print(s, len(s))
print(type({}), type(set()))
print(set("hello"), set([1, 1, 2]), set())

s.add(10); s.update([20, 30])
s.discard(999)            # no error if missing
s.remove(10)              # KeyError if missing
print(s, s.pop() in {1, 2, 3, 20, 30})
print(2 in s)             # O(1) membership – sets are hash tables

try:
    {[1, 2]}
except TypeError as e:
    print("TypeError:", e)

{1, 2, 3} 3
<class 'dict'> <class 'set'>
{'e', 'h', 'l', 'o'} {1, 2} set()
{2, 3, 20, 30} True
True
TypeError: unhashable type: 'list'


### Set algebra (Venn diagrams in code)

| Operation | Operator | Method |
|---|---|---|
| Union | `A \| B` | `A.union(B)` |
| Intersection | `A & B` | `A.intersection(B)` |
| Difference | `A - B` | `A.difference(B)` |
| Symmetric diff | `A ^ B` | `A.symmetric_difference(B)` |
| Subset | `A <= B` | `A.issubset(B)` |
| Superset | `A >= B` | `A.issuperset(B)` |
| Disjoint | | `A.isdisjoint(B)` |

In [5]:
A = {1, 2, 3, 4}
B = {3, 4, 5, 6}
print(A | B, A & B, A - B, B - A, A ^ B)
print({1, 2} <= A, A >= {1, 2}, A.isdisjoint({9}))
print(A.union([7, 8]))        # methods accept any iterable, operators need sets

# Practical
print("duplicates removed:", list(set([1, 2, 2, 3, 3, 3])))
print("common friends:", {"amy", "raj", "kim"} & {"raj", "lee", "kim"})

fs = frozenset([1, 2, 3])     # immutable set -> hashable
d = {fs: "usable as key"}
print(d[frozenset([3, 2, 1])])

# Set comprehension
print({x * x for x in range(-3, 4)})

{1, 2, 3, 4, 5, 6} {3, 4} {1, 2} {5, 6} {1, 2, 5, 6}
True True True
{1, 2, 3, 4, 7, 8}
duplicates removed: [1, 2, 3]
common friends: {'kim', 'raj'}
usable as key
{0, 9, 4, 1}


---
# PART C — DICTIONARIES

A dict maps **keys → values**. Since Python 3.7 it **preserves insertion order**. Keys must be **hashable/immutable** (str, int, tuple of immutables…); values can be anything. Lookup by key is O(1) on average.

In [6]:
student = {"name": "Asha", "age": 19, "marks": [88, 92]}
d2 = dict(name="Ravi", age=21)                  # keyword form
d3 = dict([("a", 1), ("b", 2)])                 # from pairs
d4 = dict.fromkeys(["x", "y"], 0)               # same value for all keys
d5 = {k: k ** 2 for k in range(5)}              # comprehension
print(student, d2, d3, d4, d5, sep="\n")

# Access
print(student["name"])
print(student.get("email"))                     # None instead of KeyError
print(student.get("email", "not given"))
try:
    student["email"]
except KeyError as e:
    print("KeyError:", e)

# Add / update / delete
student["email"] = "a@x.com"
student["age"] += 1
del student["marks"]
print(student.pop("email"), student)
print(student.pop("zzz", "no such key"))
print("name" in student, "Asha" in student)     # `in` checks KEYS

{'name': 'Asha', 'age': 19, 'marks': [88, 92]}
{'name': 'Ravi', 'age': 21}
{'a': 1, 'b': 2}
{'x': 0, 'y': 0}
{0: 0, 1: 1, 2: 4, 3: 9, 4: 16}
Asha
None
not given
KeyError: 'email'
a@x.com {'name': 'Asha', 'age': 20}
no such key
True False


### Dict methods you must know

| Method | Purpose |
|---|---|
| `d.keys()` `d.values()` `d.items()` | live *views* of keys / values / (key, value) pairs |
| `d.get(k, default)` | safe lookup |
| `d.setdefault(k, default)` | get, and insert default if missing |
| `d.update(other)` | merge / overwrite from another dict or pairs |
| `d.pop(k[, default])` | remove key & return value |
| `d.popitem()` | remove & return the *last* inserted pair (LIFO) |
| `d.clear()` `d.copy()` | empty / shallow copy |
| `dict.fromkeys(keys, v)` | build dict |
| `d1 \| d2`, `d1 \|= d2` | merge operators (3.9+) |

In [7]:
d = {"a": 1, "b": 2, "c": 3}
print(list(d.keys()), list(d.values()), list(d.items()))

for k in d:
    print(k, end=" ")
print()
for k, v in d.items():
    print(f"{k}={v}", end="; ")
print()

print(d.setdefault("d", 4), d.setdefault("a", 100), d)
d.update({"a": 10, "e": 5})
d.update(f=6, g=7)
print(d)
print(d.popitem(), len(d))
print({"x": 1} | {"y": 2, "x": 99})          # right side wins

# Sort a dict by value / key
scores = {"Asha": 88, "Ravi": 95, "Meena": 70}
print(sorted(scores.items(), key=lambda kv: kv[1], reverse=True))
print(dict(sorted(scores.items())))
print(max(scores, key=scores.get), min(scores.values()))

# Invert a dict
print({v: k for k, v in scores.items()})

# Views are live
keys = scores.keys()
scores["New"] = 1
print(list(keys))

['a', 'b', 'c'] [1, 2, 3] [('a', 1), ('b', 2), ('c', 3)]
a b c 
a=1; b=2; c=3; 
4 1 {'a': 1, 'b': 2, 'c': 3, 'd': 4}
{'a': 10, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7}
('g', 7) 6
{'x': 99, 'y': 2}
[('Ravi', 95), ('Asha', 88), ('Meena', 70)]
{'Asha': 88, 'Meena': 70, 'Ravi': 95}
Ravi 70
{88: 'Asha', 95: 'Ravi', 70: 'Meena'}
['Asha', 'Ravi', 'Meena', 'New']


### Nested dicts & JSON-like data

In [8]:
school = {
    "class_A": {"Asha": 88, "Ravi": 92},
    "class_B": {"Tom": 75},
}
print(school["class_A"]["Ravi"])
school["class_B"]["Sam"] = 60
for cls, kids in school.items():
    print(cls, "avg =", sum(kids.values()) / len(kids))

import json
print(json.dumps(school, indent=2))

92
class_A avg = 90.0
class_B avg = 67.5
{
  "class_A": {
    "Asha": 88,
    "Ravi": 92
  },
  "class_B": {
    "Tom": 75,
    "Sam": 60
  }
}


### Counting & grouping — the 3 patterns you'll use forever

In [9]:
text = "to be or not to be that is the question"
words = text.split()

# 1) manual counting with get()
freq = {}
for w in words:
    freq[w] = freq.get(w, 0) + 1
print(freq)

# 2) Counter
from collections import Counter, defaultdict
c = Counter(words)
print(c.most_common(3), c["be"], c["zzz"])

# 3) grouping with defaultdict(list)
groups = defaultdict(list)
for w in words:
    groups[len(w)].append(w)
print(dict(groups))

# defaultdict(int) as a counter
cnt = defaultdict(int)
for ch in "banana":
    cnt[ch] += 1
print(dict(cnt))

# Counter arithmetic
print(Counter("aab") + Counter("abc"), Counter("aabb") - Counter("ab"))

{'to': 2, 'be': 2, 'or': 1, 'not': 1, 'that': 1, 'is': 1, 'the': 1, 'question': 1}
[('to', 2), ('be', 2), ('or', 1)] 2 0
{2: ['to', 'be', 'or', 'to', 'be', 'is'], 3: ['not', 'the'], 4: ['that'], 8: ['question']}
{'b': 1, 'a': 3, 'n': 2}
Counter({'a': 3, 'b': 2, 'c': 1}) Counter({'a': 1, 'b': 1})


## Hashing: why keys must be immutable
A dict stores each key at a slot computed from `hash(key)`. If a key could change, its hash would change and the dict would lose it.

In [10]:
print(hash("abc") == hash("abc"), hash(42), hash((1, 2)))
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("TypeError:", e)
print(hash(1) == hash(1.0) == hash(True))        # equal numbers hash equal
d = {1: "a"}
d[1.0] = "b"; d[True] = "c"
print(d)                                         # 1, 1.0 and True are the SAME key!

True 42 -3550055125485641917
TypeError: unhashable type: 'list'
True
{1: 'c'}


## Comparison table (memorise!)

| Feature | **List** | **Tuple** | **Set** | **Dict** |
|---|---|---|---|---|
| Syntax | `[1,2]` | `(1,2)` | `{1,2}` | `{"a":1}` |
| Ordered | ✅ | ✅ | ❌ | ✅ (insertion) |
| Mutable | ✅ | ❌ | ✅ | ✅ |
| Duplicates | ✅ | ✅ | ❌ | keys ❌ / values ✅ |
| Indexed | by position | by position | ❌ | by key |
| Hashable | ❌ | ✅ (if items are) | ❌ (frozenset ✅) | ❌ |
| Lookup `x in c` | O(n) | O(n) | O(1) | O(1) on keys |
| Use when… | changing sequence | fixed record | uniqueness / fast lookup / set maths | mapping by name |

## Real-world uses
- **Phone book, config, JSON, caches, counters** → dict
- **Remove duplicates, membership tests, unique visitors** → set
- **Coordinates, DB rows, function returns** → tuple
- **Ordered, changing data** → list

---
# 🕵️ Hidden Facts & Pro Tricks

In [11]:
# 1. Merge dicts three ways
a, b = {"x": 1}, {"y": 2}
print({**a, **b}, a | b)

# 2. dict(zip(keys, values)) — the quickest way to build a dict
print(dict(zip("abc", range(3))))

# 3. Membership on a LIST of 100k items vs a SET
import timeit
big_list = list(range(100_000)); big_set = set(big_list)
t_list = timeit.timeit("99_999 in big_list", globals=globals(), number=200)
t_set = timeit.timeit("99_999 in big_set", globals=globals(), number=200)
print("set is faster by ~", round(t_list / t_set), "x")

# 4. Dict remembers insertion order (since 3.7 guaranteed)
d = {}
d["z"] = 1; d["a"] = 2
print(list(d))

# 5. Set operations on dict views: keys() behave like sets
d1, d2 = {"a": 1, "b": 2}, {"b": 3, "c": 4}
print(d1.keys() & d2.keys(), d1.keys() - d2.keys())

# 6. Safe nested access
cfg = {"db": {"host": "localhost"}}
print(cfg.get("db", {}).get("port", 5432))

# 7. Swap keys and values, group values by key in ONE pass with setdefault
pairs = [("fruit", "apple"), ("veg", "kale"), ("fruit", "fig")]
g = {}
for k, v in pairs:
    g.setdefault(k, []).append(v)
print(g)

# 8. Dict as a switch / lookup table
grade_of = {range(90, 101): "A", range(80, 90): "B"}
score = 85
print(next((g for r, g in grade_of.items() if score in r), "C"))

# 9. Tuple compare and sort by multiple fields
people = [("Ravi", 21), ("Asha", 21), ("Tom", 19)]
print(sorted(people, key=lambda p: (p[1], p[0])))

# 10. Star-unpack in function calls & dict unpack for kwargs
def area(w, h): return w * h
print(area(*(3, 4)), area(**{"w": 2, "h": 5}))

# 11. Deduplicate while keeping order
print(list(dict.fromkeys([3, 1, 3, 2, 1])))

# 12. Set of tuples to remove duplicate rows; frozenset for set-of-sets
print({(1, 2), (1, 2), (2, 1)})

# 13. The "count things" one-liner
print({ch: "hello world".count(ch) for ch in set("hello world")})

# 14. Tuple immutability trick: `+=` on a tuple's list item raises AND changes it!
t = ([1], 2)
try:
    t[0] += [9]
except TypeError as e:
    print("TypeError but ->", t)

{'x': 1, 'y': 2} {'x': 1, 'y': 2}
{'a': 0, 'b': 1, 'c': 2}
set is faster by ~ 22349 x
['z', 'a']
{'b'} {'a'}
5432
{'fruit': ['apple', 'fig'], 'veg': ['kale']}
B
[('Tom', 19), ('Asha', 21), ('Ravi', 21)]
12 10
[3, 1, 2]
{(1, 2), (2, 1)}
{' ': 1, 'h': 1, 'r': 1, 'd': 1, 'e': 1, 'o': 2, 'w': 1, 'l': 3}
TypeError but -> ([1, 9], 2)


**Hidden fact:** Python dicts got ~25% smaller and became ordered in 3.6 thanks to a redesign ("compact dict") by Raymond Hettiger — order preservation was originally a *side effect*.

**Hidden fact:** an empty dict `{}` and `dict()` differ in speed: literal `{}` is faster because it skips a function call. Same for `[]` vs `list()`.

**Hidden fact:** set/dict lookup is O(1) *on average* but O(n) in the worst case (many hash collisions).

---
# 📝 Exam Corner

**Theory**
1. Differentiate list, tuple, set, dictionary (use the table).
2. Why are tuples faster than lists? Why is `(5)` not a tuple?
3. Which types can be dictionary keys? Why?
4. Explain `get()`, `setdefault()`, `update()`, `pop()`, `popitem()`.
5. Write set operations for A = {1,2,3}, B = {3,4}.
6. Explain shallow vs deep copy in dicts.

**Output prediction**

In [12]:
d = {"a": 1, "b": 2}
d["c"] = d.get("c", 0) + 5
d.update({"a": 100})
print(d, len(d))
print(list(d.items())[0])
s = {1, 2, 3}
t = {3, 4}
print(s & t, s | t, s - t, s ^ t)
print(type((1)), type((1,)), type({}), type({1}))
print({1, 2, 2, 3} == {3, 2, 1})
x = dict.fromkeys("ab", [])
x["a"].append(1)
print(x)

{'a': 100, 'b': 2, 'c': 5} 3
('a', 100)
{3} {1, 2, 3, 4} {1, 2} {1, 2, 4}
<class 'int'> <class 'tuple'> <class 'dict'> <class 'set'>
True
{'a': [1], 'b': [1]}


<details><summary>💡 Answers</summary>

```
{'a': 100, 'b': 2, 'c': 5} 3
('a', 100)
{3} {1, 2, 3, 4} {1, 2} {1, 2, 4}
<class 'int'> <class 'tuple'> <class 'dict'> <class 'set'>
True
{'a': [1], 'b': [1]}
```
The last one: `fromkeys` puts the **same list object** under every key.
</details>

---
## ✅ Summary
- Tuple: immutable ordered record; set: unique unordered, O(1) membership + algebra; dict: hash map of key→value.
- Keys/set items must be hashable. `{}` is a dict; `set()` is empty set; `(5,)` is a 1-tuple.
- Learn `get`, `setdefault`, `items`, `update`, `Counter`, `defaultdict`.

➡️ **Next:** Module 08 — Functions.